In [1]:
import os
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.mamba import Mamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
class BiMambaWrapper(nn.Module):
    def __init__(self, d_model, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state)
        self.mamba_fwd = Mamba(config)
        self.mamba_bwd = Mamba(config)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        y_fwd = self.mamba_fwd(x)
        y_bwd = torch.flip(self.mamba_bwd(torch.flip(x, dims=[1])), dims=[1])
        return self.norm(y_fwd + y_bwd)


class ROIPatchEmbed(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64):
        super().__init__()
        self.grid_size = roi_size // patch_size
        self.n_tokens = (self.grid_size ** 3) * n_rois
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_tokens, d_model) * 0.02)

    def forward(self, rois):
        B, N = rois.shape[0], rois.shape[1]
        toks = [self.patch_conv(rois[:, i]).flatten(2).transpose(1, 2) for i in range(N)]
        return torch.cat(toks, dim=1) + self.pos_embed


class VisionMambaBranch(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64, n_layers=2, d_state=16):
        super().__init__()
        self.patch_embed = ROIPatchEmbed(n_rois, roi_size, patch_size, d_model)
        self.bimamba = BiMambaWrapper(d_model, n_layers, d_state)

    def forward(self, rois):
        tokens = self.patch_embed(rois)
        tokens = self.bimamba(tokens)
        return tokens.mean(dim=1)


class VisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, rois):
        pooled = self.branch(rois)
        return self.classifier(self.dropout(pooled))


class MultimodalVisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=64,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_rois, pet_rois):
        fused = torch.cat([self.mri_branch(mri_rois), self.pet_branch(pet_rois)], dim=1)
        return self.classifier(self.dropout(fused))

In [3]:
COHORT_CSV    = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG = "D:/mamba_model/preprocessed_cache_roi64_aug"
PET_CACHE_AUG = "D:/mamba_model/preprocessed_cache_pet_aug"
CKPT_DIR      = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(COHORT_CSV)
sessions = df["mri_session"].values
labels   = df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv
)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 126 | Val: 42 | Test: 42


In [4]:
class ROIDataset(Dataset):
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples = []
        self.cache_dir = cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        rois = np.load(f"{self.cache_dir}/{key}_{version}.npy").astype(np.float32)
        return torch.tensor(rois).unsqueeze(1), torch.tensor(label, dtype=torch.long)


class MultimodalROIDataset(Dataset):
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir = mri_cache_dir
        self.pet_cache_dir = pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_rois = np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy").astype(np.float32)
        pet_rois = np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy").astype(np.float32)
        return (torch.tensor(mri_rois).unsqueeze(1), torch.tensor(pet_rois).unsqueeze(1),
                torch.tensor(label, dtype=torch.long))

In [5]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for rois, labels in loader:
        rois, labels = rois.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(rois)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for rois, labels in loader:
            rois, labels = rois.to(device), labels.to(device)
            outputs = model(rois)
            total_loss += criterion(outputs, labels).item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    tpr = recall_score(all_labels, all_preds, zero_division=0)
    tnr = specificity_score(all_labels, all_preds)
    return avg_loss, acc, tpr, tnr


def train_epoch_mm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mri_rois, pet_rois, labels in loader:
        mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(mri_rois, pet_rois)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_mm(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for mri_rois, pet_rois, labels in loader:
            mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
            outputs = model(mri_rois, pet_rois)
            total_loss += criterion(outputs, labels).item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    tpr = recall_score(all_labels, all_preds, zero_division=0)
    tnr = specificity_score(all_labels, all_preds)
    return avg_loss, acc, tpr, tnr

In [6]:
def measure_inference_time(model, loader, device, is_multimodal, n_batches_to_time=20):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches_to_time:
                break
            if is_multimodal:
                mri_rois, pet_rois, labels = batch
                mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
                batch_size = mri_rois.shape[0]
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                t0 = time.time()
                _ = model(mri_rois, pet_rois)
            else:
                rois, labels = batch
                rois = rois.to(device)
                batch_size = rois.shape[0]
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                t0 = time.time()
                _ = model(rois)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            times.append((time.time() - t0) / batch_size)
    return np.mean(times), np.std(times)


def try_compute_flops(model, loader, device, is_multimodal):
    try:
        model.eval()
        sample_batch = next(iter(loader))
        with torch.no_grad():
            if is_multimodal:
                mri_rois, pet_rois, labels = sample_batch
                sample_input = (mri_rois[:1].to(device), pet_rois[:1].to(device))
                macs, params = profile(model, inputs=sample_input, verbose=False)
            else:
                rois, labels = sample_batch
                sample_input = rois[:1].to(device)
                macs, params = profile(model, inputs=(sample_input,), verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs estimation failed: {e})")
        return None


def run_one_seed(seed, model_class, train_loader, val_loader, test_loader,
                  is_multimodal, save_prefix, max_epochs=101, patience=15):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = model_class(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

    train_fn = train_epoch_mm if is_multimodal else train_epoch
    eval_fn = evaluate_mm if is_multimodal else evaluate

    best_val_loss = float("inf")
    no_improvement = 0
    best_epoch = 0
    save_path = f"{CKPT_DIR}/{save_prefix}_seed{seed}.pt"
    total_train_time = 0

    print(f"\n--- Seed {seed} ---")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
    print("-" * 70)

    for epoch in range(1, max_epochs):
        t0 = time.time()
        train_loss = train_fn(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_tpr, val_tnr = eval_fn(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        epoch_time = time.time() - t0
        total_train_time += epoch_time

        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
              f"{val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            no_improvement = 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improvement += 1
            if no_improvement >= patience:
                print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    test_loss, test_acc, test_tpr, test_tnr = eval_fn(model, test_loader, criterion, device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_time_mean, inf_time_std = measure_inference_time(model, test_loader, device, is_multimodal)
    flops = try_compute_flops(model, test_loader, device, is_multimodal)

    flops_str = f"{flops/1e9:.2f}GFLOPs" if flops else "N/A"
    print(f"\n  >>> Seed {seed} TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}% | "
          f"train_time={total_train_time/60:.1f}min | inf={inf_time_mean*1000:.2f}ms | {flops_str}")

    return {"seed": seed, "acc": test_acc, "tpr": test_tpr, "tnr": test_tnr,
            "best_epoch": best_epoch, "train_time_sec": total_train_time,
            "n_params": n_params, "inf_time_ms": inf_time_mean * 1000,
            "flops": flops}

In [7]:
mri_train_dataset = ROIDataset(X_train, y_train, MRI_CACHE_AUG, is_mri=True, is_train=True)
mri_val_dataset   = ROIDataset(X_val,   y_val,   MRI_CACHE_AUG, is_mri=True, is_train=False)
mri_test_dataset  = ROIDataset(X_test,  y_test,  MRI_CACHE_AUG, is_mri=True, is_train=False)

BATCH_SIZE = 4
mri_train_loader = DataLoader(mri_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
mri_val_loader   = DataLoader(mri_val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
mri_test_loader  = DataLoader(mri_test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("=== MRI-ONLY: 3-seed run ===")
mri_seed_results = []
for seed in [1, 7, 123]:
    result = run_one_seed(seed, VisionMambaModel, mri_train_loader, mri_val_loader, mri_test_loader,
                           is_multimodal=False, save_prefix="vim_mri_d32")
    mri_seed_results.append(result)

mri_accs = [r["acc"] for r in mri_seed_results]
mri_tprs = [r["tpr"] for r in mri_seed_results]
mri_tnrs = [r["tnr"] for r in mri_seed_results]
mri_times = [r["train_time_sec"] for r in mri_seed_results]

print(f"\nMRI-ONLY SUMMARY (3 seeds)")
print(f"Acc={np.mean(mri_accs)*100:.1f}±{np.std(mri_accs)*100:.1f}%")
print(f"TPR={np.mean(mri_tprs)*100:.1f}±{np.std(mri_tprs)*100:.1f}%")
print(f"TNR={np.mean(mri_tnrs)*100:.1f}±{np.std(mri_tnrs)*100:.1f}%")
print(f"Avg train time: {np.mean(mri_times)/60:.1f} min/seed")

=== MRI-ONLY: 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7039 |     0.6934 |   0.5000 |   0.0000 |   1.0000 |  10.8s
     2 |     0.6936 |     0.6910 |   0.5000 |   1.0000 |   0.0000 |  10.5s
     3 |     0.6986 |     0.6892 |   0.6190 |   0.5238 |   0.7143 |  10.4s
     4 |     0.7000 |     0.6876 |   0.5952 |   0.9048 |   0.2857 |  10.1s
     5 |     0.6785 |     0.6868 |   0.5000 |   1.0000 |   0.0000 |  10.0s
     6 |     0.6869 |     0.6841 |   0.5476 |   0.5238 |   0.5714 |   9.9s
     7 |     0.6842 |     0.6838 |   0.5000 |   1.0000 |   0.0000 |   9.8s
     8 |     0.6778 |     0.6966 |   0.5000 |   1.0000 |   0.0000 |  10.0s
     9 |     0.6753 |     0.6814 |   0.5238 |   1.0000 |   0.0476 |  10.5s
    10 |     0.6676 |     0.6788 |   0.5714 |   0.9524 |   0.1905 |  10.8s
    11 |     0.6632 |     0.6749 |   0.6667 |   0.4762 |   

In [8]:
pet_train_dataset = ROIDataset(X_train, y_train, PET_CACHE_AUG, is_mri=False, is_train=True)
pet_val_dataset   = ROIDataset(X_val,   y_val,   PET_CACHE_AUG, is_mri=False, is_train=False)
pet_test_dataset  = ROIDataset(X_test,  y_test,  PET_CACHE_AUG, is_mri=False, is_train=False)

pet_train_loader = DataLoader(pet_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
pet_val_loader   = DataLoader(pet_val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
pet_test_loader  = DataLoader(pet_test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("=== PET-ONLY: 3-seed run ===")
pet_seed_results = []
for seed in [1, 7, 123]:
    result = run_one_seed(seed, VisionMambaModel, pet_train_loader, pet_val_loader, pet_test_loader,
                           is_multimodal=False, save_prefix="vim_pet_d32")
    pet_seed_results.append(result)

pet_accs = [r["acc"] for r in pet_seed_results]
pet_tprs = [r["tpr"] for r in pet_seed_results]
pet_tnrs = [r["tnr"] for r in pet_seed_results]
pet_times = [r["train_time_sec"] for r in pet_seed_results]

print(f"\nPET-ONLY SUMMARY (3 seeds)")
print(f"Acc={np.mean(pet_accs)*100:.1f}±{np.std(pet_accs)*100:.1f}%")
print(f"TPR={np.mean(pet_tprs)*100:.1f}±{np.std(pet_tprs)*100:.1f}%")
print(f"TNR={np.mean(pet_tnrs)*100:.1f}±{np.std(pet_tnrs)*100:.1f}%")
print(f"Avg train time: {np.mean(pet_times)/60:.1f} min/seed")

=== PET-ONLY: 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7046 |     0.6938 |   0.5000 |   0.0000 |   1.0000 |  57.2s
     2 |     0.6941 |     0.6913 |   0.5000 |   1.0000 |   0.0000 |  10.2s
     3 |     0.6972 |     0.6895 |   0.5476 |   0.4286 |   0.6667 |   9.9s
     4 |     0.7001 |     0.6881 |   0.5238 |   0.7619 |   0.2857 |  10.9s
     5 |     0.6760 |     0.6876 |   0.4762 |   0.8571 |   0.0952 |  10.5s
     6 |     0.6860 |     0.6855 |   0.5714 |   0.3810 |   0.7619 |  10.3s
     7 |     0.6822 |     0.6860 |   0.4762 |   0.9524 |   0.0000 |  10.5s
     8 |     0.6753 |     0.6991 |   0.5000 |   1.0000 |   0.0000 |  10.4s
     9 |     0.6742 |     0.6841 |   0.4762 |   0.9524 |   0.0000 |  11.0s


KeyboardInterrupt: 

In [ ]:
mm_train_dataset = MultimodalROIDataset(X_train, y_train, MRI_CACHE_AUG, PET_CACHE_AUG, is_train=True)
mm_val_dataset   = MultimodalROIDataset(X_val,   y_val,   MRI_CACHE_AUG, PET_CACHE_AUG, is_train=False)
mm_test_dataset  = MultimodalROIDataset(X_test,  y_test,  MRI_CACHE_AUG, PET_CACHE_AUG, is_train=False)

mm_train_loader = DataLoader(mm_train_dataset, batch_size=4, shuffle=True,  num_workers=0)
mm_val_loader   = DataLoader(mm_val_dataset,   batch_size=4, shuffle=False, num_workers=0)
mm_test_loader  = DataLoader(mm_test_dataset,  batch_size=4, shuffle=False, num_workers=0)

print("=== MULTIMODAL: 3-seed run ===")
mm_seed_results = []
for seed in [1, 7, 123]:
    result = run_one_seed(seed, MultimodalVisionMambaModel, mm_train_loader, mm_val_loader, mm_test_loader,
                           is_multimodal=True, save_prefix="vim_mm_d32")
    mm_seed_results.append(result)

mm_accs = [r["acc"] for r in mm_seed_results]
mm_tprs = [r["tpr"] for r in mm_seed_results]
mm_tnrs = [r["tnr"] for r in mm_seed_results]
mm_times = [r["train_time_sec"] for r in mm_seed_results]

print(f"\nMULTIMODAL SUMMARY (3 seeds)")
print(f"Acc={np.mean(mm_accs)*100:.1f}±{np.std(mm_accs)*100:.1f}%")
print(f"TPR={np.mean(mm_tprs)*100:.1f}±{np.std(mm_tprs)*100:.1f}%")
print(f"TNR={np.mean(mm_tnrs)*100:.1f}±{np.std(mm_tnrs)*100:.1f}%")
print(f"Avg train time: {np.mean(mm_times)/60:.1f} min/seed")

In [ ]:
print(f"{'Model':<22} | {'Accuracy':>14} | {'TPR':>14} | {'TNR':>14}")
print("-" * 75)
print(f"{'MNA-net (Vo et al.)':<22} | {'82.9%':>14} | {'85.7%':>14} | {'80.0%':>14}")
print(f"{'MRI-only (3-seed)':<22} | {np.mean(mri_accs)*100:>6.1f}±{np.std(mri_accs)*100:<5.1f}% | "
      f"{np.mean(mri_tprs)*100:>6.1f}±{np.std(mri_tprs)*100:<5.1f}% | "
      f"{np.mean(mri_tnrs)*100:>6.1f}±{np.std(mri_tnrs)*100:<5.1f}%")
print(f"{'PET-only (3-seed)':<22} | {np.mean(pet_accs)*100:>6.1f}±{np.std(pet_accs)*100:<5.1f}% | "
      f"{np.mean(pet_tprs)*100:>6.1f}±{np.std(pet_tprs)*100:<5.1f}% | "
      f"{np.mean(pet_tnrs)*100:>6.1f}±{np.std(pet_tnrs)*100:<5.1f}%")
print(f"{'Multimodal (3-seed)':<22} | {np.mean(mm_accs)*100:>6.1f}±{np.std(mm_accs)*100:<5.1f}% | "
      f"{np.mean(mm_tprs)*100:>6.1f}±{np.std(mm_tprs)*100:<5.1f}% | "
      f"{np.mean(mm_tnrs)*100:>6.1f}±{np.std(mm_tnrs)*100:<5.1f}%")